In [ ]:
!pip install gcloud
!gcloud auth application-default login
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=EdQlI5xyO1z4q1Z8eC4hVX3lmBGHAX&prompt=consent&token_usage=remote&access_type=offline&code_challenge=KvVsIZZgEehqjFZDy-OFm6QT9J-Ya-jZCHu9kg8gp4A&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AXEQxIBvDnQHssY2qhZzqLFx7UzBcWRAweLgfVqtnokIR83g22ZKZK4G9NAmZO-JF9O_MA

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [ ]:
df = pd.read_excel('/content/Base_MUNIC_2024_20251107.xlsx', sheet_name='Informática e comunicação')
df


,Cod Munic,Uf,Cod Uf,Desc Mun,Populacao,Faixa_populacao,Regiao,Mtic011,Mtic012,Mtic013,...,Mtic367,Mtic368,Mtic37,Mtic381,Mtic382,Mtic383,Mtic384,Mtic385,Mtic386,Mtic387
0,1100015,RO,11,Alta Floresta DOeste,22853,4 - 20001 até 50000,1 - Norte,Não,Não,Sim,...,Não,Não,Não,Sim,Não,Não,Sim,Não,Não,Não
1,1100023,RO,11,Ariquemes,108573,6 - 100001 até 500000,1 - Norte,Não,Não,Sim,...,-,-,Sim,Não,Não,Não,Não,Sim,Não,Não
2,1100031,RO,11,Cabixi,5690,2 - 5001 até 10000,1 - Norte,Não,Não,Sim,...,-,-,Não,-,-,-,-,-,-,Sim
3,1100049,RO,11,Cacoal,97637,5 - 50001 até 100000,1 - Norte,Não,Não,Sim,...,-,-,Sim,Sim,Não,Não,Não,Não,Não,Não
4,1100056,RO,11,Cerejeiras,16975,3 - 10001 até 20000,1 - Norte,Não,Não,Sim,...,Não,Não,Sim,Sim,Sim,Não,Não,Não,Não,Não
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5565,5222005,GO,52,Vianópolis,15476,3 - 10001 até 20000,5 - Centro-Oeste,Não,Não,Sim,...,-,-,Sim,Sim,Sim,Sim,Não,Não,Não,Não
5566,5222054,GO,52,Vicentinópolis,9077,2 - 5001 até 10000,5 - Centro-Oeste,Não,Não,Sim,...,-,Sim,Não,-,-,-,-,-,-,Sim
5567,5222203,GO,52,Vila Boa,4185,1 - Até 5000,5 - Centro-Oeste,Não,Não,Sim,...,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Não
5568,5222302,GO,52,Vila Propício,5982,2 - 5001 até 10000,5 - Centro-Oeste,Sim,Não,Sim,...,-,-,Não,Sim,Não,Sim,Sim,Não,Não,Não


In [ ]:
df = df[['Cod Uf', 'Desc Mun', 'Cod Munic', 'Mtic12a7', 'Mtic12b13', 'Mtic12a4', 'Mtic12a5', 'Mtic12a6', 'Mtic12b1']]
df

,Cod Uf,Desc Mun,Cod Munic,Mtic12a7,Mtic12b13,Mtic12a4,Mtic12a5,Mtic12a6,Mtic12b1
0,11,Alta Floresta DOeste,1100015,Sim,Não,Sim,Sim,Sim,Sim
1,11,Ariquemes,1100023,Sim,Sim,Sim,Sim,Sim,Sim
2,11,Cabixi,1100031,Sim,Sim,Não,Sim,Sim,Sim
3,11,Cacoal,1100049,Sim,Sim,Sim,Sim,Sim,Sim
4,11,Cerejeiras,1100056,Sim,Não,Sim,Sim,Sim,Sim
...,...,...,...,...,...,...,...,...,...
5565,52,Vianópolis,5222005,Sim,Não,Sim,Sim,Sim,Sim
5566,52,Vicentinópolis,5222054,Sim,Sim,Sim,Sim,Sim,Sim
5567,52,Vila Boa,5222203,Sim,Sim,Sim,Sim,Sim,Sim
5568,52,Vila Propício,5222302,Sim,Não,Sim,Sim,Sim,Sim


In [ ]:
df = df.rename(columns={'Cod Uf': 'cod_uf',
                        'Desc Mun':'nome_municipio',
                        'Cod Munic':'id_municipio',
                        'Mtic12a7':'concursos_publicos',
                        'Mtic12b13':'pesquisa_satisfacao_servicos_estado',
                        'Mtic12a4':'diario',
                        'Mtic12a5':'legislacao',
                        'Mtic12a6':'financas',
                        'Mtic12b1':'ouvidoria_atendimento_cidadao'})

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   cod_uf                               5570 non-null   int64 
 1   nome_municipio                       5570 non-null   object
 2   id_municipio                         5570 non-null   int64 
 3   concursos_publicos                   5570 non-null   object
 4   pesquisa_satisfacao_servicos_estado  5570 non-null   object
 5   diario                               5570 non-null   object
 6   legislacao                           5570 non-null   object
 7   financas                             5570 non-null   object
 8   ouvidoria_atendimento_cidadao        5570 non-null   object
dtypes: int64(2), object(7)
memory usage: 391.8+ KB


In [ ]:
cod_uf = pd.read_csv('/content/ESTADIC_2019 - Variáveis externas.csv', sep=',')[['UF','COD_UF']]

In [ ]:
x= cod_uf.pivot_table(columns=('UF','COD_UF'), aggfunc='size')


In [ ]:
cod_uf = pd.DataFrame(x).reset_index()[['UF','COD_UF']]

In [ ]:
df = df.merge(cod_uf, right_on='COD_UF',left_on='cod_uf')
df

,cod_uf,nome_municipio,id_municipio,concursos_publicos,pesquisa_satisfacao_servicos_estado,diario,legislacao,financas,ouvidoria_atendimento_cidadao,UF,COD_UF
0,11,Alta Floresta DOeste,1100015,Sim,Não,Sim,Sim,Sim,Sim,RO,11
1,11,Ariquemes,1100023,Sim,Sim,Sim,Sim,Sim,Sim,RO,11
2,11,Cabixi,1100031,Sim,Sim,Não,Sim,Sim,Sim,RO,11
3,11,Cacoal,1100049,Sim,Sim,Sim,Sim,Sim,Sim,RO,11
4,11,Cerejeiras,1100056,Sim,Não,Sim,Sim,Sim,Sim,RO,11
...,...,...,...,...,...,...,...,...,...,...,...
5565,52,Vianópolis,5222005,Sim,Não,Sim,Sim,Sim,Sim,GO,52
5566,52,Vicentinópolis,5222054,Sim,Sim,Sim,Sim,Sim,Sim,GO,52
5567,52,Vila Boa,5222203,Sim,Sim,Sim,Sim,Sim,Sim,GO,52
5568,52,Vila Propício,5222302,Sim,Não,Sim,Sim,Sim,Sim,GO,52


In [ ]:
df['ano']= 2024

In [ ]:
df.columns

Index(['cod_uf', 'nome_municipio', 'id_municipio', 'concursos_publicos',
       'pesquisa_satisfacao_servicos_estado', 'diario', 'legislacao',
       'financas', 'ouvidoria_atendimento_cidadao', 'UF', 'COD_UF', 'ano'],
      dtype='object')

In [ ]:
df = df[['ano','cod_uf','UF', 'id_municipio','nome_municipio','concursos_publicos',
       'pesquisa_satisfacao_servicos_estado',
       'ouvidoria_atendimento_cidadao', 'diario', 'legislacao', 'financas']]

In [ ]:
df

,ano,cod_uf,UF,id_municipio,nome_municipio,concursos_publicos,pesquisa_satisfacao_servicos_estado,ouvidoria_atendimento_cidadao,diario,legislacao,financas
0,2024,11,RO,1100015,Alta Floresta DOeste,Sim,Não,Sim,Sim,Sim,Sim
1,2024,11,RO,1100023,Ariquemes,Sim,Sim,Sim,Sim,Sim,Sim
2,2024,11,RO,1100031,Cabixi,Sim,Sim,Sim,Não,Sim,Sim
3,2024,11,RO,1100049,Cacoal,Sim,Sim,Sim,Sim,Sim,Sim
4,2024,11,RO,1100056,Cerejeiras,Sim,Não,Sim,Sim,Sim,Sim
...,...,...,...,...,...,...,...,...,...,...,...
5565,2024,52,GO,5222005,Vianópolis,Sim,Não,Sim,Sim,Sim,Sim
5566,2024,52,GO,5222054,Vicentinópolis,Sim,Sim,Sim,Sim,Sim,Sim
5567,2024,52,GO,5222203,Vila Boa,Sim,Sim,Sim,Sim,Sim,Sim
5568,2024,52,GO,5222302,Vila Propício,Sim,Não,Sim,Sim,Sim,Sim


In [ ]:
df['concursos_publicos']=np.where(df['concursos_publicos']=='-','Sem dados',df['concursos_publicos'])
df['concursos_publicos']=np.where(df['concursos_publicos']=='Não informou','Sem dados',df['concursos_publicos'])
df['concursos_publicos']=np.where(df['concursos_publicos']=='Recusa','Sem dados',df['concursos_publicos'])

In [ ]:
df['pesquisa_satisfacao_servicos_estado']=np.where(df['pesquisa_satisfacao_servicos_estado']=='-','Sem dados',df['pesquisa_satisfacao_servicos_estado'])
df['pesquisa_satisfacao_servicos_estado']=np.where(df['pesquisa_satisfacao_servicos_estado']=='Não informou','Sem dados',df['pesquisa_satisfacao_servicos_estado'])
df['pesquisa_satisfacao_servicos_estado']=np.where(df['pesquisa_satisfacao_servicos_estado']=='Recusa','Sem dados',df['pesquisa_satisfacao_servicos_estado'])

In [ ]:
df['diario']=np.where(df['diario']=='-','Sem dados',df['diario'])
df['diario']=np.where(df['diario']=='Não informou','Sem dados',df['diario'])
df['diario']=np.where(df['diario']=='Recusa','Sem dados',df['diario'])

In [ ]:
df['legislacao']=np.where(df['legislacao']=='-','Sem dados',df['legislacao'])
df['legislacao']=np.where(df['legislacao']=='Não informou','Sem dados',df['legislacao'])
df['legislacao']=np.where(df['legislacao']=='Recusa','Sem dados',df['legislacao'])

In [ ]:
df['financas']=np.where(df['financas']=='-','Sem dados',df['financas'])
df['financas']=np.where(df['financas']=='Não informou','Sem dados',df['financas'])
df['financas']=np.where(df['financas']=='Recusa','Sem dados',df['financas'])

In [ ]:
df['ouvidoria_atendimento_cidadao']=np.where(df['ouvidoria_atendimento_cidadao']=='-','Sem dados',df['ouvidoria_atendimento_cidadao'])
df['ouvidoria_atendimento_cidadao']=np.where(df['ouvidoria_atendimento_cidadao']=='Não informou','Sem dados',df['ouvidoria_atendimento_cidadao'])
df['ouvidoria_atendimento_cidadao']=np.where(df['ouvidoria_atendimento_cidadao']=='Recusa','Sem dados',df['ouvidoria_atendimento_cidadao'])

In [ ]:
df

,ano,cod_uf,UF,id_municipio,nome_municipio,concursos_publicos,pesquisa_satisfacao_servicos_estado,ouvidoria_atendimento_cidadao,diario,legislacao,financas
0,2024,11,RO,1100015,Alta Floresta DOeste,Sim,Não,Sim,Sim,Sim,Sim
1,2024,11,RO,1100023,Ariquemes,Sim,Sim,Sim,Sim,Sim,Sim
2,2024,11,RO,1100031,Cabixi,Sim,Sim,Sim,Não,Sim,Sim
3,2024,11,RO,1100049,Cacoal,Sim,Sim,Sim,Sim,Sim,Sim
4,2024,11,RO,1100056,Cerejeiras,Sim,Não,Sim,Sim,Sim,Sim
...,...,...,...,...,...,...,...,...,...,...,...
5565,2024,52,GO,5222005,Vianópolis,Sim,Não,Sim,Sim,Sim,Sim
5566,2024,52,GO,5222054,Vicentinópolis,Sim,Sim,Sim,Sim,Sim,Sim
5567,2024,52,GO,5222203,Vila Boa,Sim,Sim,Sim,Sim,Sim,Sim
5568,2024,52,GO,5222302,Vila Propício,Sim,Não,Sim,Sim,Sim,Sim


In [ ]:
df = df.rename(columns={'codigo_uf':'cod_uf',
                       'UF':'sigla_uf'})

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 11 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  5570 non-null   int64 
 1   cod_uf                               5570 non-null   int64 
 2   sigla_uf                             5570 non-null   object
 3   id_municipio                         5570 non-null   int64 
 4   nome_municipio                       5570 non-null   object
 5   concursos_publicos                   5570 non-null   object
 6   pesquisa_satisfacao_servicos_estado  5570 non-null   object
 7   ouvidoria_atendimento_cidadao        5570 non-null   object
 8   diario                               5570 non-null   object
 9   legislacao                           5570 non-null   object
 10  financas                             5570 non-null   object
dtypes: int64(3), object(8)
memory usage: 478.8+

In [ ]:
df['pesquisa_satisfacao_servicos_estado'].unique()

array(['Não', 'Sim', 'Sem dados'], dtype=object)

# Consumindo o ano de 2019 através do GBQ

In [ ]:


query = """SELECT * FROM `repositoriodedadosgpsp.participacao_transparencia.MUNIC_transparencia_internet` WHERE ano = 2019"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_2019 = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')



Downloading: 100%|██████████|


In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  5570 non-null   Int64 
 1   cod_uf                               5570 non-null   Int64 
 2   sigla_uf                             5570 non-null   object
 3   id_municipio                         5570 non-null   Int64 
 4   nome_municipio                       5570 non-null   object
 5   concursos_publicos                   5570 non-null   object
 6   pesquisa_satisfacao_servicos_estado  5570 non-null   object
 7   diario_legislacao_financas           5570 non-null   object
 8   ouvidoria_atendimento_cidadao        5570 non-null   object
dtypes: Int64(3), object(6)
memory usage: 408.1+ KB


In [ ]:
for col in ['diario', 'legislacao', 'financas']:
    df_2019[col] = df_2019['diario_legislacao_financas']

In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  5570 non-null   Int64 
 1   cod_uf                               5570 non-null   Int64 
 2   sigla_uf                             5570 non-null   object
 3   id_municipio                         5570 non-null   Int64 
 4   nome_municipio                       5570 non-null   object
 5   concursos_publicos                   5570 non-null   object
 6   pesquisa_satisfacao_servicos_estado  5570 non-null   object
 7   diario_legislacao_financas           5570 non-null   object
 8   ouvidoria_atendimento_cidadao        5570 non-null   object
 9   diario                               5570 non-null   object
 10  legislacao                           5570 non-null   object
 11  financas                             5570 n

In [ ]:
df_2019.head(10)

,ano,cod_uf,sigla_uf,id_municipio,nome_municipio,concursos_publicos,pesquisa_satisfacao_servicos_estado,diario_legislacao_financas,ouvidoria_atendimento_cidadao,diario,legislacao,financas
0,2019,11,RO,1100015,Alta Floresta D'Oeste,Sim,Sim,Sim,Sim,Sim,Sim,Sim
1,2019,11,RO,1100023,Ariquemes,Sim,Não,Sim,Sim,Sim,Sim,Sim
2,2019,11,RO,1100031,Cabixi,Sim,Não,Sim,Sim,Sim,Sim,Sim
3,2019,11,RO,1100049,Cacoal,Sim,Não,Sim,Sim,Sim,Sim,Sim
4,2019,11,RO,1100056,Cerejeiras,Sim,Não,Sim,Sim,Sim,Sim,Sim
5,2019,11,RO,1100080,Costa Marques,Sim,Não,Sim,Sim,Sim,Sim,Sim
6,2019,11,RO,1100098,Espigão D'Oeste,Sim,Não,Sim,Sim,Sim,Sim,Sim
7,2019,11,RO,1100106,Guajará-Mirim,Não,Não,Sim,Sim,Sim,Sim,Sim
8,2019,11,RO,1100114,Jaru,Sim,Não,Sim,Sim,Sim,Sim,Sim
9,2019,11,RO,1100122,Ji-Paraná,Sim,Não,Sim,Sim,Sim,Sim,Sim


In [ ]:
df_2019 = df_2019.drop(columns=['diario_legislacao_financas'])

In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 11 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  5570 non-null   Int64 
 1   cod_uf                               5570 non-null   Int64 
 2   sigla_uf                             5570 non-null   object
 3   id_municipio                         5570 non-null   Int64 
 4   nome_municipio                       5570 non-null   object
 5   concursos_publicos                   5570 non-null   object
 6   pesquisa_satisfacao_servicos_estado  5570 non-null   object
 7   ouvidoria_atendimento_cidadao        5570 non-null   object
 8   diario                               5570 non-null   object
 9   legislacao                           5570 non-null   object
 10  financas                             5570 non-null   object
dtypes: Int64(3), object(8)
memory usage: 495.1+

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 11 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  5570 non-null   int64 
 1   cod_uf                               5570 non-null   int64 
 2   sigla_uf                             5570 non-null   object
 3   id_municipio                         5570 non-null   int64 
 4   nome_municipio                       5570 non-null   object
 5   concursos_publicos                   5570 non-null   object
 6   pesquisa_satisfacao_servicos_estado  5570 non-null   object
 7   ouvidoria_atendimento_cidadao        5570 non-null   object
 8   diario                               5570 non-null   object
 9   legislacao                           5570 non-null   object
 10  financas                             5570 non-null   object
dtypes: int64(3), object(8)
memory usage: 478.8+

In [ ]:
df_final = pd.concat([df, df_2019], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11140 entries, 0 to 11139
Data columns (total 11 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  11140 non-null  Int64 
 1   cod_uf                               11140 non-null  Int64 
 2   sigla_uf                             11140 non-null  object
 3   id_municipio                         11140 non-null  Int64 
 4   nome_municipio                       11140 non-null  object
 5   concursos_publicos                   11140 non-null  object
 6   pesquisa_satisfacao_servicos_estado  11140 non-null  object
 7   ouvidoria_atendimento_cidadao        11140 non-null  object
 8   diario                               11140 non-null  object
 9   legislacao                           11140 non-null  object
 10  financas                             11140 non-null  object
dtypes: Int64(3), object(8)
memory usage: 990.

# Subindo para o GBQ

In [ ]:
# Define the BigQuery table schema with Portuguese descriptions
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano de referência da observação'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('sigla_uf','STRING',description='Sigla da Unidade da Federação'),
        bigquery.SchemaField('nome_municipio','STRING',description='Nome do município da observação'),
        bigquery.SchemaField('id_municipio','INTEGER',description='Identificador do município pelo IBGE'),
        bigquery.SchemaField('concursos_publicos','STRING',description='Serviços disponibilizados na internet sobre concursos públicos'),
        bigquery.SchemaField('pesquisa_satisfacao_servicos_estado','STRING',description='Serviços disponibilizados na internet sobre pesquisa de satisfação relacionada aos serviços prestados pelo estado'),
        bigquery.SchemaField('diario','STRING',description='Serviços disponibilizados na internet sobre diário oficial'),
        bigquery.SchemaField('legislacao','STRING',description='Serviços disponibilizados na internet sobre legislação estadual'),
        bigquery.SchemaField('financas','STRING',description='Serviços disponibilizados na internet sobre finanças públicas'),
        bigquery.SchemaField('ouvidoria_atendimento_cidadao','STRING',description='Serviços disponibilizados na internet sobre ouvidoria e serviços de atendimento ao cidadão'),
]

# Initialize BigQuery client connection
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create reference to target dataset
dataset_ref = client.dataset('participacao_transparencia')

# Create reference to target table with standardized naming convention:
# FONTE_algo_intuitivo_dado (MUNIC_quantidade_vinculos_mapa_v1)
table_ref = dataset_ref.table('MUNIC_transparencia_internet_v1')

# Configure the load job with our schema definition
job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Optional parameters (commented out):
    # write_disposition="WRITE_TRUNCATE",  # Overwrites table if exists
    # create_disposition="CREATE_IF_NEEDED"  # Default behavior
)

# Execute the load job to upload DataFrame to BigQuery
job = client.load_table_from_dataframe(
    dataframe=df_final,
    destination=table_ref,
    job_config=job_config
)

# Wait for the job to complete
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=250ef8f7-0ecd-4244-8a6b-18d183159e0b>

Subindo para o GBQ